# Phase 1. 강건성 테스트 리포트

1차 EfficientNetV2S 모델(threshold 0.24, 재학습 없음)에 JPEG 재압축·리사이즈·블러를 적용한 결과입니다.

- 평가셋: 현재 `data/test` (1차 ID Test와 같은 26,876장)
- 변환은 256x256으로 맞춘 뒤 적용
- 소스별 fake/real 수가 달라서 소스 비교는 **balanced accuracy**와 **AUC** 기준

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

RESULT_DIR = Path('../outputs/exp2/robustness')
summary = pd.read_csv(RESULT_DIR / 'robustness_summary.csv')
ORDER = ['none', 'jpeg_q90', 'jpeg_q70', 'jpeg_q50', 'resize_0.75', 'resize_0.5', 'blur_0.5', 'blur_1.0']

## 1. 전체 결과

In [ ]:
columns = ['perturbation', 'accuracy', 'balanced_accuracy', 'roc_auc_real_positive', 'fake_recall', 'real_recall', 'delta_balanced_accuracy']
summary[summary['subset'] == 'all'][columns].set_index('perturbation').loc[ORDER].round(4)

## 2. 소스별 balanced accuracy / Fake recall

In [ ]:
for metric in ['balanced_accuracy', 'fake_recall', 'roc_auc_real_positive']:
    table = summary.pivot(index='perturbation', columns='subset', values=metric).loc[ORDER]
    display(table.round(4))
    ax = table.drop(columns='all').plot(marker='o', figsize=(9, 4), title=metric)
    ax.set_xticks(range(len(ORDER)), ORDER, rotation=30)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f'robustness_{metric}.png', dpi=150)
    plt.show()

## 3. 원본 해상도 분포

fake와 real의 원본 해상도가 소스 안에서 다르면, 모델이 '진짜/가짜' 대신 해상도 차이(리사이즈 흔적)를 단서로 쓸 수 있습니다.

In [ ]:
sizes = pd.read_csv(RESULT_DIR / 'image_sizes.csv')
sizes.pivot_table(index=['source', 'size'], columns='label', values='images', fill_value=0)

## 4. GRAVEX에서 놓친 Fake

GRAVEX fake에는 전체 생성 얼굴뿐 아니라 실제 영상의 얼굴만 바꾼 **부분 조작**(FaceForensics++, CelebDF, DFDC)이 섞여 있습니다. 원본 파일명 기준 출처별 놓친 비율:

In [ ]:
pd.read_csv(RESULT_DIR / 'gradcam_gravex' / 'gravex_miss_rate_by_origin.csv')

### Grad-CAM

위: 원본, 아래: fake 쪽 logit에 기여한 영역

In [ ]:
display(Image(RESULT_DIR / 'gradcam_gravex' / 'missed_top.png'))
display(Image(RESULT_DIR / 'gradcam_gravex' / 'caught_examples.png'))